In [4]:
import pymupdf


def pdf_to_markdown(pdf_path, markdown_path):
    

    markdown = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(page.get_text())
            markdown.append("\n\n")

    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown))

    print(f"Markdown file created: {markdown_path}")


pdf_to_markdown("Sherlock Holmes.pdf", "output.md")


Markdown file created: output.md


In [ ]:
from pathlib import Path
import re
 
INPUT_FILE = Path("output.md")
OUTPUT_FOLDER = Path("dataset")
 

BOOK_RANGES = [
    ("A Study in Scarlet", 13, 68),
    ("The Sign of the Four", 73, 122),
    ("The Adventures of Sherlock Holmes", 129, 282),
    ("The Memoirs of Sherlock Holmes", 287, 417),
    ("The Return of Sherlock Holmes", 423, 586),
    ("The Hound of the Baskervilles", 591, 662),
    ("The Valley of Fear", 669, 742),
    ("His Last Bow", 751, 852),
    ("The Case-Book of Sherlock Holmes", 859, 987),
]

def get_book_name(page_number):
    
    for book_name, first_page, last_page in BOOK_RANGES:
        if first_page <= page_number <= last_page:
            return book_name
 
    return None
 
 
def clean_text(text):
    
    text = re.sub(r'\s+', ' ', text)
    text = text.replace(r'[\r\n]+', '\n').strip()
 
    return text
 
 
def split_pages():

    text = INPUT_FILE.read_text(encoding="utf-8")
 
    pages = re.split(r"^##\s*Page\s+(\d+)\s*$", text, flags=re.MULTILINE)
 
    OUTPUT_FOLDER.mkdir(exist_ok=True)
 
    for i in range(1, len(pages), 2):
 
        page_number = int(pages[i])
        page_content = clean_text(pages[i + 1])
        book_name = get_book_name(page_number)
 
        if book_name:
 
            markdown = (
                f"# {book_name}\n\n"
                f"## Page {page_number}\n\n"
                f"{page_content}"
            )
 
            file_name = f"{book_name} - Page {page_number}.md"
            (OUTPUT_FOLDER / file_name).write_text(markdown, encoding="utf-8")
 
 
split_pages()

In [ ]:
from sentence_transformers import SentenceTransformer
from pathlib import Path

DATASET_FOLDER = Path("dataset")
MODEL_NAME = "intfloat/multilingual-e5-large"


def read_page(file_path):
    
    lines = file_path.read_text(encoding="utf-8").splitlines()

    book_name = lines[0].replace("# ", "")
    page_number = int(lines[2].replace("## Page ", ""))
    content = " ".join(lines[3:]).strip()

    return {
        "book_name": book_name,
        "page_number": page_number,
        "content": content,
    }



files = sorted(DATASET_FOLDER.glob("*.md"))
pages = [read_page(file) for file in files]
texts = [f"passage: {page['content']}" for page in pages]

model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()



/opt/anaconda3/envs/Ai/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 30/30 [01:49<00:00,  3.65s/it]


In [16]:
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv
import os
import time

load_dotenv()
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION")


client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=120,         
)

collection_name = QDRANT_COLLECTION
vector_size = len(embeddings[0])

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE,
        ),
    )


points = [
    models.PointStruct(
        id=index,
        vector=embedding,
        payload=page,
    )
    for index, (page, embedding) in enumerate(zip(pages, embeddings))
]

batch_size = 32          
for start in range(0, len(points), batch_size):
    batch = points[start:start + batch_size]

    for attempt in range(3):                
        try:
            client.upsert(
                collection_name=collection_name,
                points=batch,
                wait=False,                  
            )
            break
        except Exception as e:
            if attempt == 2:
                raise
            print(f"Batch {start} failed ({e}); retrying...")
            time.sleep(2)

   

print(f"Uploaded {len(points)} pages to Qdrant.")

Uploaded 932 pages to Qdrant.


In [66]:

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

query = input("Ask a question: " )

router_llm = ChatGroq(

    model=os.getenv("GROQ_MODEL"),
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

SYSTEM_PROMPT = """You classify messages for a Sherlock Holmes book search system.
Return exactly one label and nothing else:
retrieve - questions about the books, characters, places, or events
chitchat - greetings, thanks, or casual conversation, and in this case you can answer the question
off-topic - anything unrelated to the books, and tell the user that you are only answering questions about the Sherlock Holmes books"""

router_messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=query),
]


route = router_llm.invoke(router_messages).content.strip().lower()
route = route.splitlines()[0].strip(" `.,:")

if route not in {"retrieve", "chitchat", "off-topic"}:
    route = "off-topic"

print("Route:", route)


Route: retrieve


In [67]:

load_dotenv()
top_k = 5

if route == "retrieve":
    query_vector = model.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
    )[0].tolist()

    results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=top_k,
    ).points


    context = ""
    for result in results:
        page = result.payload
        context += (
            f"Book: {page['book_name']}\n"
            f"Page: {page['page_number']}\n"
            f"Content: {page['content']}\n\n"
        )

        print("Score:", result.score)
        print("Book:", page["book_name"])
        print("Page:", page["page_number"])
        print("Content:", page["content"])
        print("-" * 80)
        
else:
    print("No database search needed.")


Score: 0.849542
Book: The Adventures of Sherlock Holmes
Page: 141
Content: The Red-Headed League
--------------------------------------------------------------------------------
Score: 0.8440887
Book: The Adventures of Sherlock Holmes
Page: 145
Content: The Red-Headed League “The ﬁrst thing that put us out was that advertise- ment. Spaulding, he came down into the ofﬁce just this day eight weeks, with this very paper in his hand, and he says: “ ‘I wish to the Lord, Mr. Wilson, that I was a red-headed man.’ “ ‘Why that?’ I asks. “ ‘Why,’ says he, ‘here’s another vacancy on the League of the Red-headed Men. It’s worth quite a little fortune to any man who gets it, and I under- stand that there are more vacancies than there are men, so that the trustees are at their wits’ end what to do with the money. If my hair would only change colour, here’s a nice little crib all ready for me to step into.’ “ ‘Why, what is it, then?’ I asked. You see, Mr. Holmes, I am a very stay-at-home man, and as 

In [ ]:
if route == "retrieve":
    keyword_query = "Mrs. Hudson"
    
    keywords = keyword_query.lower().split()
    keyword_results = []

    for page in pages:
        content = page["content"].lower()
        score = sum(content.count(keyword) for keyword in keywords)r

        if score > 0:
            keyword_results.append({
                "score": score,
                "book_name": page["book_name"],
                "page_number": page["page_number"],
                "content": page["content"],
            })

    keyword_results.sort(key=lambda result: result["score"], reverse=True)

    for result in keyword_results[:top_k]:
        print("Keyword score:", result["score"])
        print("Book:", result["book_name"])
        print("Page:", result["page_number"])
        print("Content:", result["content"])
        print("-" * 80)


Keyword score: 12
Book: The Sign of the Four
Page: 101
Content: The Sign of the Four CHAPTER IX. A Break in the Chain It was late in the afternoon before I woke, strengthened and refreshed. Sherlock Holmes still sat exactly as I had left him, save that he had laid aside his violin and was deep in a book. He looked across at me, as I stirred, and I noticed that his face was dark and troubled. “You have slept soundly,” he said. “I feared that our talk would wake you.” “I heard nothing,” I answered. “Have you had fresh news, then?” “Unfortunately, no. I confess that I am surprised and disappointed. I expected something deﬁnite by this time. Wiggins has just been up to report. He says that no trace can be found of the launch. It is a provoking check, for every hour is of importance.” “Can I do anything? I am perfectly fresh now, and quite ready for another night’s outing.” “No, we can do nothing. We can only wait. If we go ourselves, the message might come in our absence, and delay be caus

In [68]:
if route == "retrieve":
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.messages import SystemMessage, HumanMessage

    gemini_llm = ChatGoogleGenerativeAI(
        model=os.getenv("GEMINI_MODEL"),
        api_key=os.getenv("GEMINI_API_KEY"),
        temperature=0,
    )

    messages = [
        SystemMessage(
            content="Answer only from the provided pages. If the answer is not there, say you do not know. Keep the answer concise."
        ),
        HumanMessage(
            content=f"Context:\n{context}\nQuestion:\n{query}"
        ),
    ]

    response = gemini_llm.invoke(messages)
    print("\nAnswer:")
    print(response.text)


Answer:
According to the provided text, the League of the Red-headed Men was founded by an American millionaire named Ezekiah Hopkins. Hopkins was red-headed and had great sympathy for all red-headed men; upon his death, he left his fortune in the hands of trustees with instructions to apply the interest to provide "easy berths" (jobs) to men with bright, flame-colored hair.


In [85]:
evaluation_cases = [
    {
        "query": "Why does Sherlock Holmes always call Irene Adler the woman?",
        "relevant_pages": {129},
    },
    {
        "query": "What creature was the speckled band that came through the ventilator?",
        "relevant_pages": {229},
    },
    {
        "query": "How did the typewriter provide a clue in A Case of Identity?",
        "relevant_pages": {162},
    },
    
    {
        "query": "Why is Professor Moriarty called the Napoleon of crime?",
        "relevant_pages": {410},
    }
    
]

top_k = 2
precision_scores = []
recall_scores = []
retrieved_for_evaluation = []

for case in evaluation_cases:
    query_vector = model.encode(
        [f"query: {case['query']}"],
        normalize_embeddings=True,
    )[0].tolist()

    search_results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    ).points

    retrieved_pages = {result.payload["page_number"] for result in search_results}
    relevant_pages = case["relevant_pages"]
    relevant_retrieved = retrieved_pages & relevant_pages

    precision = len(relevant_retrieved) / len(retrieved_pages) if retrieved_pages else 0
    recall = len(relevant_retrieved) / len(relevant_pages)

    precision_scores.append(precision)
    recall_scores.append(recall)
    retrieved_for_evaluation.append((case, search_results))

    print(case["query"])
    print("Expected pages:", relevant_pages)
    print("Retrieved pages:", retrieved_pages)
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print("-" * 60)

print(f"Average precision: {sum(precision_scores) / len(precision_scores):.2f}")
print(f"Average recall:    {sum(recall_scores) / len(recall_scores):.2f}")

Why does Sherlock Holmes always call Irene Adler the woman?
Expected pages: {129}
Retrieved pages: {129, 139}
Precision: 0.50
Recall:    1.00
------------------------------------------------------------
What creature was the speckled band that came through the ventilator?
Expected pages: {229}
Retrieved pages: {229, 230}
Precision: 0.50
Recall:    1.00
------------------------------------------------------------
How did the typewriter provide a clue in A Case of Identity?
Expected pages: {162}
Retrieved pages: {162, 163}
Precision: 0.50
Recall:    1.00
------------------------------------------------------------
Why is Professor Moriarty called the Napoleon of crime?
Expected pages: {410}
Retrieved pages: {410, 675}
Precision: 0.50
Recall:    1.00
------------------------------------------------------------
Average precision: 0.50
Average recall:    1.00


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

gemini_llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL"),
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0,
)

for case, search_results in retrieved_for_evaluation:
    context = "\n\n".join(
        f"Page {result.payload['page_number']}: {result.payload['content']}"
        for result in search_results
    )

    answer = gemini_llm.invoke([
        SystemMessage(content="Answer only from the provided context. If the answer is not there, say you do not know."),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{case['query']}"),
    ]).text

    judge = gemini_llm.invoke([
        SystemMessage(content="""You are an evaluator for a question-answering system.
                                Judge the answer using only the context.
                                Return exactly this format:
                                Score: X/5
                                Grounded: yes or no
                                Reason: one short sentence"""),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{case['query']}\n\nAnswer:\n{answer}"),
    ]).text

    print("Question:", case["query"])
    print("Answer:", answer)
    print("Judge:", judge)
    print("-" * 60)

Question: Why does Sherlock Holmes always call Irene Adler the woman?
Answer: According to the provided text, to Sherlock Holmes, Irene Adler "eclipses and predominates the whole of her sex." Additionally, the text notes that while he used to make merry over the cleverness of women, he has not done so of late, and when he refers to her, it is under the "honourable title of the woman."
Judge: Score: 5/5
Grounded: yes
Reason: The answer accurately reflects the text's explanation for why Holmes refers to Irene Adler in that specific way.
------------------------------------------------------------
Question: What creature was the speckled band that came through the ventilator?
Answer: The creature was a swamp adder, described as the deadliest snake in India.
Judge: Score: 5/5
Grounded: yes
Reason: The text explicitly identifies the creature as a swamp adder, the deadliest snake in India.
------------------------------------------------------------
Question: How did the typewriter provide a